In [16]:
import numpy as np 
import json 
import nltk
from nltk.util import ngrams
from nltk import FreqDist
from collections import Counter
import math

Experiment 1 — Corpus statistics

In [2]:
documents = [] 
with open("D:\\VietAnd\\Việt Anh\\NLP_HUS\\c4-train.00000-of-01024-30K.json\\c4-train.00000-of-01024-30K.json", 'r', encoding = 'utf-8') as f: 
    for line in f: 
        data = json.loads(line) 
        documents.append(data['text']) 
print(f"Number of documents: {len(documents)}")

Number of documents: 30000


In [9]:
sentents = [] 
for document in documents: 
    sentents.extend(nltk.sent_tokenize(document))

print(f"Number of sentences: {len(sentents)}") 

tokens = [] 
for sent in sentents: 
    tokens.extend(nltk.word_tokenize(sent)) 

print(f"Number of tokens: {len(tokens)}") 

vocabulary = set(tokens)
print(f"Vocabulary size: {len(vocabulary)}") 

unigrams = list(ngrams(tokens, 1)) 
bigrams = list(ngrams(tokens, 2)) 
trigrams = list(ngrams(tokens, 3)) 
print(f"Number of unigrams: {len(unigrams)}") 
print(f"Number of bigrams: {len(bigrams)}") 
print(f"Number of trigrams: {len(trigrams)}") 

uni_fre = FreqDist(unigrams) 
bi_fre = FreqDist(bigrams)
tri_fre = FreqDist(trigrams) 

uni_once = sum(1 for count in uni_fre.values() if count == 1) 
bi_once = sum(1 for count in bi_fre.values() if count == 1) 
tri_once = sum(1 for count in tri_fre.values() if count == 1) 

print(f"Number of unigrams that appear only once: {uni_once}") 
print(f"Number of bigrams that appear only once: {bi_once}") 
print(f"Number of trigrams that appear only once: {tri_once}")

Number of sentences: 622194
Number of tokens: 12459342
Vocabulary size: 307578
Number of unigrams: 12459342
Number of bigrams: 12459341
Number of trigrams: 12459340
Number of unigrams that appear only once: 170441
Number of bigrams that appear only once: 2242028
Number of trigrams that appear only once: 6762788


In [12]:
print("Top 5 unigrams:") 
for word, count in uni_fre.most_common(5):
    print(f"{word[0]}: {count}")
print("Top 5 bigrams:")
for bigram, count in bi_fre.most_common(5):
    print(f"{bigram[0]} {bigram[1]}: {count}")
print("Top 5 trigrams:")
for trigram, count in tri_fre.most_common(5):
    print(f"{trigram[0]} {trigram[1]} {trigram[2]}: {count}")

Top 5 unigrams:
.: 560563
,: 518219
the: 497278
and: 315309
to: 300905
Top 5 bigrams:
of the: 60603
. The: 51838
, and: 46036
in the: 42924
’ s: 41225
Top 5 trigrams:
. If you: 6255
it ’ s: 4903
. It is: 4414
don ’ t: 4333
as well as: 4151


EXPERIMENT 2

In [17]:
import random 
random.seed(42) 
random.shuffle(documents) 
n = len(documents) 
train_end = int(0.8 * n) 
valid_end = int(0.9 * n) 

train_documents = documents[:train_end] 
valid_documents = documents[train_end:valid_end] 
test_documents = documents[valid_end:] 

print(f"Len of train_documents: {len(train_documents)}") 
print(f"Len of valid_documents: {len(valid_documents)}") 
print(f"Len of test_documents: {len(test_documents)}") 


def tokenize_documents(documents): 
    corpus = [] 
    for document in documents: 
        sentences = nltk.sent_tokenize(document) 
        for sentence in sentences: 
            tokens = nltk.word_tokenize(sentence) 

            if len(tokens) > 0: 
                corpus.append(tokens) 


    return corpus 

train_corpus = tokenize_documents(train_documents) 
valid_corpus = tokenize_documents(valid_documents)
test_corpus = tokenize_documents(test_documents) 

def add_sentence_markers(corpus): 
    result = [] 
    for sentence in corpus: 

        sentence = (
            ["<s>"] + sentence + ["</s>"]
        )
        result.append(sentence) 
    return result 

train_corpus = add_sentence_markers(train_corpus)
valid_corpus = add_sentence_markers(valid_corpus)
test_corpus = add_sentence_markers(test_corpus)

def buil_vocabulary(corpus): 
    vocabulary = set() 
    for sentence in corpus: 
        for word in sentence: 
            vocabulary.add(word) 

    vocabulary.add("<UNK>") 
    return vocabulary 

vocabulary = buil_vocabulary(train_corpus) 

def replace_unknow_words(corpus, vocabulary): 
    result = [] 
    for sentence in corpus: 
        new_sentence = [] 
        for word in sentence: 
            if word in vocabulary: 
                new_sentence.append(word) 
            else: 
                new_sentence.append("<UNK>") 
        result.append(new_sentence) 
    return result
train_corpus = replace_unknow_words(train_corpus, vocabulary)
valid_corpus = replace_unknow_words(valid_corpus, vocabulary) 
test_corpus = replace_unknow_words(test_documents, vocabulary) 

def count_unigrams(corpus): 
    counts = Counter() 
    for sentence in corpus: 
        for word in sentence: 
            counts[word] += 1
    return counts 

def count_bigrams(corpus): 
    counts = Counter() 
    for sentence in corpus: 
        for i in range(len(sentence) - 1): 
            bigram = (sentence[i], sentence[i + 1]) 
            counts[bigram] += 1 
    return counts

def count_trigrams(corpus): 
    counts = Counter() 
    for sentence in corpus: 
        for i in range(len(sentence) - 2): 
            trigram = (sentence[i], sentence[i + 1], sentence[i + 2]) 
            counts[trigram] += 1
    return counts


unigram_counts = count_unigrams(train_corpus)

bigram_counts = count_bigrams(train_corpus)

trigram_counts = count_trigrams(train_corpus)

def bigram_mle_probability(context, word, unigram_counts, bigram_counts): 
    denominator = unigram_counts[context] 
    if denominator == 0: 
        return 0.0 
    numerator = bigram_counts[(context, word)]
    return numerator / denominator 

def bigram_laplace_probability(
    context,
    word,
    unigram_counts,
    bigram_counts,
    vocabulary
):

    V = len(vocabulary)

    numerator = (
        bigram_counts[(context, word)]
        + 1
    )

    denominator = (
        unigram_counts[context]
        + V
    )

    return numerator / denominator

def trigram_mle_probability(
    context,
    word,
    bigram_counts,
    trigram_counts
):

    w1, w2 = context

    denominator = bigram_counts[
        (w1, w2)
    ]

    if denominator == 0:
        return 0.0

    numerator = trigram_counts[
        (w1, w2, word)
    ]

    return numerator / denominator

def trigram_laplace_probability(
    context,
    word,
    bigram_counts,
    trigram_counts,
    vocabulary
):

    w1, w2 = context

    V = len(vocabulary)

    numerator = (
        trigram_counts[
            (w1, w2, word)
        ] + 1
    )

    denominator = (
        bigram_counts[
            (w1, w2)
        ] + V
    )

    return numerator / denominator

def sentence_probability_bigram(
    sentence,
    probability_function
):

    probability = 1.0

    for i in range(1, len(sentence)):

        context = sentence[i - 1]

        word = sentence[i]

        p = probability_function(
            context,
            word
        )

        probability *= p

    return probability



def perplexity_bigram(
    corpus,
    probability_function
):

    log_probability = 0.0
    N = 0

    for sentence in corpus:

        for i in range(1, len(sentence)):

            context = sentence[i - 1]
            word = sentence[i]

            p = probability_function(
                context,
                word
            )

            if p <= 0:
                return float("inf")

            log_probability += math.log(p)

            N += 1

    return math.exp(
        -log_probability / N
    )

def perplexity_trigram(
    corpus,
    probability_function
):

    log_probability = 0.0
    N = 0

    for sentence in corpus:

        for i in range(2, len(sentence)):

            context = (
                sentence[i - 2],
                sentence[i - 1]
            )

            word = sentence[i]

            p = probability_function(
                context,
                word
            )

            if p <= 0:
                return float("inf")

            log_probability += math.log(p)

            N += 1

    return math.exp(
        -log_probability / N
    )

Len of train_documents: 24000
Len of valid_documents: 3000
Len of test_documents: 3000


In [18]:
bigram_mle = lambda context, word: bigram_mle_probability(
    context,
    word,
    unigram_counts,
    bigram_counts
)
train_ppl_bigram_mle = perplexity_bigram(
    train_corpus,
    bigram_mle
)

valid_ppl_bigram_mle = perplexity_bigram(
    valid_corpus,
    bigram_mle
)

test_ppl_bigram_mle = perplexity_bigram(
    test_corpus,
    bigram_mle
)

bigram_laplace = lambda context, word: bigram_laplace_probability(
    context,
    word,
    unigram_counts,
    bigram_counts,
    vocabulary
)
train_ppl_bigram_laplace = perplexity_bigram(
    train_corpus,
    bigram_laplace
)

valid_ppl_bigram_laplace = perplexity_bigram(
    valid_corpus,
    bigram_laplace
)

test_ppl_bigram_laplace = perplexity_bigram(
    test_corpus,
    bigram_laplace
)

trigram_mle = lambda context, word: trigram_mle_probability(
    context,
    word,
    bigram_counts,
    trigram_counts
)

train_ppl_trigram_mle = perplexity_trigram(
    train_corpus,
    trigram_mle
)

valid_ppl_trigram_mle = perplexity_trigram(
    valid_corpus,
    trigram_mle
)

test_ppl_trigram_mle = perplexity_trigram(
    test_corpus,
    trigram_mle
)

trigram_laplace = lambda context, word: trigram_laplace_probability(
    context,
    word,
    bigram_counts,
    trigram_counts,
    vocabulary
)
train_ppl_trigram_laplace = perplexity_trigram(
    train_corpus,
    trigram_laplace
)

valid_ppl_trigram_laplace = perplexity_trigram(
    valid_corpus,
    trigram_laplace
)

test_ppl_trigram_laplace = perplexity_trigram(
    test_corpus,
    trigram_laplace
)

In [19]:
print("\nExperiment 2 — MLE vs Laplace")
print("-" * 60)

print(
    f"{'Model':<25}"
    f"{'Train PPL':>12}"
    f"{'Valid PPL':>12}"
    f"{'Test PPL':>12}"
)

print("-" * 60)

print(
    f"{'Bigram MLE':<25}"
    f"{train_ppl_bigram_mle:>12.4f}"
    f"{valid_ppl_bigram_mle:>12.4f}"
    f"{test_ppl_bigram_mle:>12.4f}"
)

print(
    f"{'Bigram Laplace':<25}"
    f"{train_ppl_bigram_laplace:>12.4f}"
    f"{valid_ppl_bigram_laplace:>12.4f}"
    f"{test_ppl_bigram_laplace:>12.4f}"
)

print(
    f"{'Trigram MLE':<25}"
    f"{train_ppl_trigram_mle:>12.4f}"
    f"{valid_ppl_trigram_mle:>12.4f}"
    f"{test_ppl_trigram_mle:>12.4f}"
)

print(
    f"{'Trigram Laplace':<25}"
    f"{train_ppl_trigram_laplace:>12.4f}"
    f"{valid_ppl_trigram_laplace:>12.4f}"
    f"{test_ppl_trigram_laplace:>12.4f}"
)

print("-" * 60)


Experiment 2 — MLE vs Laplace
------------------------------------------------------------
Model                       Train PPL   Valid PPL    Test PPL
------------------------------------------------------------
Bigram MLE                   118.5887         inf         inf
Bigram Laplace              5379.2892   6938.8451 187080.1735
Trigram MLE                    9.9314         inf         inf
Trigram Laplace            49422.3656  82461.3238 261417.6090
------------------------------------------------------------


## 12. Giải thích kết quả Experiment 2 — MLE vs Laplace Smoothing

### 12.1. Kết quả thực nghiệm

| Model | Train PPL | Valid PPL | Test PPL |
|---|---:|---:|---:|
| Bigram MLE | 118.5887 | `inf` | `inf` |
| Bigram Laplace | 5379.2892 | 6938.8451 | 187080.1735 |
| Trigram MLE | 9.9314 | `inf` | `inf` |
| Trigram Laplace | 49422.3656 | 82461.3238 | 261417.6090 |

---

### 12.2. Tại sao Bigram MLE và Trigram MLE có PPL = `inf` trên Validation/Test?

Với MLE, xác suất Bigram được tính theo:

\[
P(w_i|w_{i-1})
=
\frac{C(w_{i-1},w_i)}
{C(w_{i-1})}
\]

Trong khi đó, xác suất Trigram được tính theo:

\[
P(w_i|w_{i-2},w_{i-1})
=
\frac{
C(w_{i-2},w_{i-1},w_i)
}{
C(w_{i-2},w_{i-1})
}
\]

Nếu một Bigram hoặc Trigram chưa từng xuất hiện trong tập training thì:

\[
C(N\text{-gram})=0
\]

Do đó:

\[
P(N\text{-gram})=0
\]

Khi tính Perplexity, ta sử dụng log probability:

\[
PPL =
\exp\left(
-\frac{1}{N}
\sum_i \log P(w_i|context)
\right)
\]

Mà:

\[
\log(0)=-\infty
\]

nên:

\[
PPL=\infty
\]

Vì vậy, MLE có thể cho `inf` trên validation/test khi xuất hiện những N-gram chưa từng xuất hiện trong training.

Đây là vấn đề được gọi là **zero-frequency problem**.

---

### 12.3. Tại sao MLE trên Training không bị `inf`?

Model được xây dựng từ chính tập training.

Những N-gram được sử dụng để tính PPL trên training đều là những N-gram đã xuất hiện trong quá trình đếm, nên chúng có count lớn hơn 0 và xác suất MLE lớn hơn 0.

Do đó:

```text
Bigram MLE:
Train PPL = 118.5887

Trigram MLE:
Train PPL = 9.9314